# Lab 2 – Exercise 1 (Groceries)

**Student**: 22521609 – Phạm Duy Tuấn  

In [2]:
import itertools
from itertools import combinations
from collections import Counter

import pandas as pd

# mlxtend for Apriori, FP-Growth, and association rules
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori as mlxtend_apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## Question 1

**Question 1 (from lab sheet)**  
*"Read the `groceries.csv` dataset and convert it into a list of transactions. Print the number of transactions and the number of items."*

**Explanation**  
Each row in `groceries.csv` represents a market basket (a transaction).  
We will read the file line by line, split by commas, and store each row as a Python list of items.  
All transactions together will be stored in a list called `transactions`.

In [3]:
# Question 1: Read groceries.csv and build list of transactions

transactions = []

file_path = "../Lab 2-20260331/groceries.csv"  # adjust path if needed

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        # Remove trailing newline and split by comma into individual items
        items = line.strip().split(",")
        # Optionally remove empty strings if any
        items = [item.strip() for item in items if item.strip()]
        if items:  # ignore completely empty lines
            transactions.append(items)

num_transactions = len(transactions)

# Count number of *distinct* items across all transactions
all_items = [item for t in transactions for item in t]
unique_items = set(all_items)
num_unique_items = len(unique_items)

print(f"Number of transactions: {num_transactions}")
print(f"Number of unique items: {num_unique_items}")

Number of transactions: 9835
Number of unique items: 169


## Question 2

**Question 2 (from lab sheet)**  
*"Compute the number of transactions, the number of unique items, and the top 10 most frequent items."*

**Explanation**  
We already computed the number of transactions and unique items.  
Here we use `collections.Counter` to count how many times each item appears in all transactions, then select the **top 10** most frequent items.

This gives us a first sense of which products are most popular in the dataset.

In [4]:
# Question 2: basic statistics and top-10 frequent items

num_transactions = len(transactions)
all_items = [item for t in transactions for item in t]
unique_items = set(all_items)
num_unique_items = len(unique_items)

print(f"Number of transactions: {num_transactions}")
print(f"Number of unique items: {num_unique_items}")

item_counter = Counter(all_items)

print("\nTop 10 most frequent items (item, count):")
for item, cnt in item_counter.most_common(10):
    print(f"{item:25s} {cnt}")

Number of transactions: 9835
Number of unique items: 169

Top 10 most frequent items (item, count):
whole milk                2513
other vegetables          1903
rolls/buns                1809
soda                      1715
yogurt                    1372
bottled water             1087
root vegetables           1072
tropical fruit            1032
shopping bags             969
sausage                   924


## Question 3

**Question 3 (from lab sheet)**  
*"Write code to compute the support of each 1-itemset."*

**Explanation**  
- A **1-itemset** is simply a single item (e.g. `{"whole milk"}`).  
- The **support** of an item is defined as:

\[ support(X) = \frac{\text{number of transactions containing X}}{\text{total number of transactions}} \]

We will loop through all unique items, count in how many transactions each item appears, and then divide by the total number of transactions.  
The result will be stored in a dictionary `support_1` with key = item name, value = support (a float between 0 and 1).

In [5]:
# Question 3: compute support for each 1-itemset

num_transactions = len(transactions)
unique_items = set(item for t in transactions for item in t)

support_1 = {}
for item in unique_items:
    count = sum(1 for t in transactions if item in t)
    support_1[item] = count / num_transactions

# Show a small sample of the results
print("Number of 1-itemsets:", len(support_1))

sample_items = list(support_1.items())[:10]
print("\nSample supports for 1-itemsets:")
for item, sup in sample_items:
    print(f"{item:25s} support = {sup:.4f}")

Number of 1-itemsets: 169

Sample supports for 1-itemsets:
ready soups               support = 0.0018
long life bakery product  support = 0.0374
liquor (appetizer)        support = 0.0079
frozen vegetables         support = 0.0481
flour                     support = 0.0174
liquor                    support = 0.0111
make up remover           support = 0.0008
pudding powder            support = 0.0023
mustard                   support = 0.0120
soups                     support = 0.0068


## Question 4

**Question 4 (from lab sheet)**  
*"Find frequent 1-itemsets given a minimum support threshold."*

**Explanation**  
- A **frequent 1-itemset** is a 1-itemset whose support is **greater than or equal to** a given `min_support`.  
- In practice, we can choose a threshold such as 0.01 (1%) as suggested in the hints.

We will:
1. Set a `min_support` value.  
2. Filter `support_1` to keep only items whose support is at least this threshold.  
3. Store the result in a dictionary `L1` (level-1 frequent itemsets).

In [6]:
# Question 4: find frequent 1-itemsets (L1) for a given min_support

min_support = 0.01  # 1%, can be changed later for analysis

L1 = {item: sup for item, sup in support_1.items() if sup >= min_support}

print(f"min_support = {min_support}")
print(f"Number of frequent 1-itemsets (L1): {len(L1)}")

# Show a few examples
for item, sup in list(L1.items())[:10]:
    print(f"{item:25s} support = {sup:.4f}")

min_support = 0.01
Number of frequent 1-itemsets (L1): 88
long life bakery product  support = 0.0374
frozen vegetables         support = 0.0481
flour                     support = 0.0174
liquor                    support = 0.0111
mustard                   support = 0.0120
onions                    support = 0.0310
coffee                    support = 0.0581
red/blush wine            support = 0.0192
butter milk               support = 0.0280
cat food                  support = 0.0233


## Question 5

**Question 5 (from lab sheet)**  
*"Write code to generate candidate 2-itemsets and find frequent 2-itemsets."*

**Explanation**  
- We first take all items that are frequent in `L1` and generate all possible **2-combinations** (pairs) using `itertools.combinations`. These are the **candidate 2-itemsets** `C2`.
- For each pair `(i, j)` in `C2`, we compute its support using the same formula as before, but checking whether both items are contained in a transaction.
- We then filter by `min_support` to obtain **frequent 2-itemsets**, stored in dictionary `L2`.

In [7]:
# Question 5: generate candidate 2-itemsets (C2) and frequent 2-itemsets (L2)

items_L1 = list(L1.keys())
C2 = list(combinations(items_L1, 2))

support_2 = {}
for pair in C2:
    # We treat transaction as a set and check if pair is a subset
    count = sum(1 for t in transactions if set(pair).issubset(t))
    support_2[pair] = count / num_transactions

L2 = {pair: sup for pair, sup in support_2.items() if sup >= min_support}

print(f"Number of candidate 2-itemsets (C2): {len(C2)}")
print(f"Number of frequent 2-itemsets (L2): {len(L2)}")

# Show a few frequent 2-itemsets
for pair, sup in list(L2.items())[:10]:
    print(f"{pair} support = {sup:.4f}")

Number of candidate 2-itemsets (C2): 3828
Number of frequent 2-itemsets (L2): 213
('long life bakery product', 'whole milk') support = 0.0135
('long life bakery product', 'other vegetables') support = 0.0107
('frozen vegetables', 'whole milk') support = 0.0204
('frozen vegetables', 'rolls/buns') support = 0.0102
('frozen vegetables', 'yogurt') support = 0.0124
('frozen vegetables', 'root vegetables') support = 0.0116
('frozen vegetables', 'other vegetables') support = 0.0178
('onions', 'whole milk') support = 0.0121
('onions', 'other vegetables') support = 0.0142
('coffee', 'whole milk') support = 0.0187


## Question 6

**Question 6 (from lab sheet)**  
*"Implement the Apriori algorithm in Python. Write a function: `apriori(transactions, min_support)`"*

**Explanation**  
We implement a **general Apriori function** that:
1. Starts with 1-itemsets:
   - Compute support for every single item.
   - Keep only those items that satisfy `min_support` \(\Rightarrow L1\).
2. Repeats for k = 2, 3, ...:
   - Generate candidate k-itemsets `Ck` from `L(k-1)` using the Apriori join step.  
   - Prune candidates whose (k−1)-subsets are not all frequent.  
   - Compute support for remaining candidates.  
   - Keep only those with support ≥ `min_support` as `Lk`.
3. Stops when no more frequent itemsets can be found.

The function returns a **list of dictionaries** `[L1, L2, L3, ...]`, where each `Lk` maps itemset tuples to their support.

In [8]:
# Question 6: Implement the Apriori algorithm in Python

def generate_candidates(prev_Lk):
    """Generate candidate (k+1)-itemsets from frequent k-itemsets (prev_Lk)."""
    prev_itemsets = list(prev_Lk.keys())
    k = len(prev_itemsets[0])  # size of itemsets in prev_Lk
    candidates = set()

    # Join step: combine itemsets that share first k-1 items
    for i in range(len(prev_itemsets)):
        for j in range(i + 1, len(prev_itemsets)):
            a = prev_itemsets[i]
            b = prev_itemsets[j]
            if a[: k - 1] == b[: k - 1]:  # they share prefix of length k-1
                candidate = tuple(sorted(set(a) | set(b)))
                if len(candidate) == k + 1:
                    candidates.add(candidate)

    return list(candidates)


def prune_candidates(candidates, prev_Lk):
    """Prune candidates whose any (k-1)-subset is not frequent."""
    prev_itemsets = set(prev_Lk.keys())
    k_plus_1 = len(next(iter(candidates))) if candidates else 0
    k = k_plus_1 - 1

    pruned = []
    for cand in candidates:
        all_subsets_frequent = True
        for subset in combinations(cand, k):
            subset = tuple(sorted(subset))
            if subset not in prev_itemsets:
                all_subsets_frequent = False
                break
        if all_subsets_frequent:
            pruned.append(cand)
    return pruned


def apriori(transactions, min_support):
    """Apriori algorithm.

    Parameters
    ----------
    transactions : list of list of str
        Each inner list is a transaction containing items.
    min_support : float
        Minimum support threshold (between 0 and 1).

    Returns
    -------
    list[dict]
        A list [L1, L2, ..., Lk] of dictionaries.
        Each Lk maps itemset (tuple of items) -> support.
    """
    num_transactions = len(transactions)

    # Step 1: frequent 1-itemsets (L1)
    item_counter = Counter(item for t in transactions for item in t)
    L1 = {}
    for item, cnt in item_counter.items():
        sup = cnt / num_transactions
        if sup >= min_support:
            L1[(item,)] = sup

    L = []
    current_Lk = L1
    k = 1

    if not current_Lk:
        return L

    L.append(current_Lk)

    # Generate L2, L3, ...
    while True:
        k += 1
        # Generate candidates Ck from L(k-1)
        candidates = generate_candidates(current_Lk)
        candidates = prune_candidates(candidates, current_Lk)

        if not candidates:
            break

        # Count support for Ck
        support_k = {}
        for cand in candidates:
            count = sum(1 for t in transactions if set(cand).issubset(t))
            sup = count / num_transactions
            if sup >= min_support:
                support_k[cand] = sup

        if not support_k:
            break

        current_Lk = support_k
        L.append(current_Lk)

    return L


# Run Apriori with min_support = 0.01 as in the hints
L_levels = apriori(transactions, min_support=0.01)

print(f"Number of levels (L1, L2, ...): {len(L_levels)}")

Number of levels (L1, L2, ...): 3


## Question 7

**Question 7 (from lab sheet)**  
*"Print intermediate steps C1, L1, C2, L2, ... of Apriori."*

**Explanation**  
Our `apriori` function above directly returns the frequent itemsets `L1, L2, ...` but it does not keep `Ck` explicitly.

To keep the notebook simple and still match the spirit of the question, we:
- Use the existing `apriori` function to get all `Lk` levels.
- Print each level `L1`, `L2`, ... with all itemsets and their supports.

(If needed, we could extend the implementation to also store `Ck`, but usually for understanding Apriori the `Lk` levels are the most important.)

In [9]:
# Question 7: print all frequent itemsets L1, L2, ... from Apriori

for i, Lk in enumerate(L_levels, start=1):
    print(f"\n===== L{i} (frequent {i}-itemsets) =====")
    for itemset, sup in Lk.items():
        print(f"{itemset}: support = {sup:.4f}")


===== L1 (frequent 1-itemsets) =====
('citrus fruit',): support = 0.0828
('semi-finished bread',): support = 0.0177
('margarine',): support = 0.0586
('tropical fruit',): support = 0.1049
('yogurt',): support = 0.1395
('coffee',): support = 0.0581
('whole milk',): support = 0.2555
('pip fruit',): support = 0.0756
('cream cheese',): support = 0.0397
('other vegetables',): support = 0.1935
('condensed milk',): support = 0.0103
('long life bakery product',): support = 0.0374
('butter',): support = 0.0554
('rolls/buns',): support = 0.1839
('UHT-milk',): support = 0.0335
('bottled beer',): support = 0.0805
('potted plants',): support = 0.0173
('white bread',): support = 0.0421
('bottled water',): support = 0.1105
('chocolate',): support = 0.0496
('curd',): support = 0.0533
('flour',): support = 0.0174
('dishes',): support = 0.0176
('beef',): support = 0.0525
('frankfurter',): support = 0.0590
('soda',): support = 0.1744
('chicken',): support = 0.0429
('sugar',): support = 0.0339
('fruit/veg

## Question 8

**Question 8 (from lab sheet)**  
*"Use `TransactionEncoder` to one-hot encode the data."*

**Explanation**  
`TransactionEncoder` from `mlxtend.preprocessing` converts a list of transactions into a 2D boolean (or 0/1) array where:
- Each **row** corresponds to a transaction.
- Each **column** corresponds to an item.
- The value is `True` (or 1) if the item appears in that transaction, otherwise `False` (or 0).

We then wrap this array in a pandas `DataFrame`, which is the required input format for the `mlxtend` implementations of Apriori and FP-Growth.

In [10]:
# Question 8: one-hot encode the data using TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df = pd.DataFrame(te_array, columns=te.columns_)

print("Shape of one-hot encoded DataFrame:", df.shape)
df.head()

Shape of one-hot encoded DataFrame: (9835, 169)


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


## Question 9

**Question 9 (from lab sheet)**  
*"Use the mlxtend library to run Apriori."*

**Explanation**  
Here we use `mlxtend.frequent_patterns.apriori` to compute frequent itemsets directly from the one-hot encoded `DataFrame` `df`.

Important parameters:
- `min_support`: minimum support threshold (same idea as before).  
- `use_colnames=True`: return itemsets as actual item **names** instead of column indices.

We can also sort the result by support to inspect the most frequent itemsets.

In [11]:
# Question 9: run Apriori using mlxtend on the one-hot encoded DataFrame

min_support_mlxtend = 0.01

frequent_itemsets_ap = mlxtend_apriori(
    df,
    min_support=min_support_mlxtend,
    use_colnames=True
)

print("Number of frequent itemsets (mlxtend Apriori):", len(frequent_itemsets_ap))
frequent_itemsets_ap.sort_values(by="support", ascending=False).head()

Number of frequent itemsets (mlxtend Apriori): 333


,support,itemsets
86,0.255516,frozenset({whole milk})
55,0.193493,frozenset({other vegetables})
66,0.183935,frozenset({rolls/buns})
75,0.174377,frozenset({soda})
87,0.139502,frozenset({yogurt})


## Question 10

**Question 10 (from lab sheet)**  
*"Use the mlxtend library to run FP-Growth."*

**Explanation**  
We now use `mlxtend.frequent_patterns.fpgrowth` on the same one-hot encoded `DataFrame`.  
The parameters are similar to Apriori: `min_support` and `use_colnames=True`.

FP-Growth is usually faster than Apriori on large datasets because it avoids generating a huge number of candidates explicitly by using a compressed tree structure (FP-tree).

In [12]:
# Question 10: run FP-Growth using mlxtend

fp_itemsets = fpgrowth(
    df,
    min_support=min_support_mlxtend,
    use_colnames=True
)

print("Number of frequent itemsets (FP-Growth):", len(fp_itemsets))
fp_itemsets.sort_values(by="support", ascending=False).head()

Number of frequent itemsets (FP-Growth): 333


,support,itemsets
6,0.255516,frozenset({whole milk})
9,0.193493,frozenset({other vegetables})
13,0.183935,frozenset({rolls/buns})
24,0.174377,frozenset({soda})
3,0.139502,frozenset({yogurt})


## Question 11

**Question 11 (from lab sheet)**  
*"Use the library to generate association rules from frequent itemsets."*

**Explanation**  
We use `mlxtend.frequent_patterns.association_rules` on the frequent itemsets (here we can use the FP-Growth result `fp_itemsets`).  
Important parameters:
- `metric`: which rule-quality measure to use (here `"confidence"`).  
- `min_threshold`: minimum value for the selected metric (e.g. `0.2`).

The function returns a `DataFrame` where each row corresponds to a rule with columns like: `antecedents`, `consequents`, `support`, `confidence`, `lift`, etc.

In [13]:
# Question 11: generate association rules from frequent itemsets

rules = association_rules(
    fp_itemsets,
    metric="confidence",
    min_threshold=0.2
)

print("Number of generated rules:", len(rules))
rules.head()

Number of generated rules: 234


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({citrus fruit}),frozenset({whole milk}),0.082766,0.255516,0.030503,0.368550,1.442377,1.0,0.009355,1.179008,0.334375,0.099108,0.151829,0.243965
1,frozenset({citrus fruit}),frozenset({yogurt}),0.082766,0.139502,0.021657,0.261671,1.875752,1.0,0.010111,1.165467,0.509009,0.107957,0.141975,0.208459
2,frozenset({citrus fruit}),frozenset({tropical fruit}),0.082766,0.104931,0.019929,0.240786,2.294702,1.0,0.011244,1.178942,0.615125,0.118788,0.151782,0.215354
3,frozenset({citrus fruit}),frozenset({other vegetables}),0.082766,0.193493,0.028876,0.348894,1.803140,1.0,0.012862,1.238674,0.485603,0.116728,0.192685,0.249066
4,frozenset({citrus fruit}),frozenset({root vegetables}),0.082766,0.108998,0.017692,0.213759,1.961121,1.0,0.008671,1.133243,0.534310,0.101636,0.117576,0.188036


## Question 12

**Question 12 (from lab sheet)**  
*"Filter the top 10 rules by confidence."*

**Explanation**  
We sort the `rules` DataFrame by the `confidence` column in **descending** order and take the first 10 rows.  
These rules have the highest conditional probability that the consequent appears when the antecedent is present.

In [14]:
# Question 12: top 10 rules by confidence

rules_sorted = rules.sort_values(by="confidence", ascending=False)

print("Top 10 rules sorted by confidence:")
rules_sorted.head(10)

Top 10 rules sorted by confidence:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
12,"frozenset({root vegetables, citrus fruit})",frozenset({other vegetables}),0.017692,0.193493,0.010371,0.586207,3.029608,1.0,0.006948,1.949059,0.681990,0.051646,0.486932,0.319903
50,"frozenset({tropical fruit, root vegetables})",frozenset({other vegetables}),0.021047,0.193493,0.012303,0.584541,3.020999,1.0,0.008231,1.941244,0.683367,0.060835,0.484867,0.324062
108,"frozenset({yogurt, curd})",frozenset({whole milk}),0.017285,0.255516,0.010066,0.582353,2.279125,1.0,0.005649,1.782567,0.571107,0.038313,0.439011,0.310874
78,"frozenset({butter, other vegetables})",frozenset({whole milk}),0.020031,0.255516,0.011490,0.573604,2.244885,1.0,0.006371,1.745992,0.565878,0.043512,0.427260,0.309285
54,"frozenset({tropical fruit, root vegetables})",frozenset({whole milk}),0.021047,0.255516,0.011998,0.570048,2.230969,1.0,0.006620,1.731553,0.563627,0.045350,0.422484,0.308502
162,"frozenset({yogurt, root vegetables})",frozenset({whole milk}),0.025826,0.255516,0.014540,0.562992,2.203354,1.0,0.007941,1.703594,0.560625,0.054497,0.413006,0.309948
222,"frozenset({domestic eggs, other vegetables})",frozenset({whole milk}),0.022267,0.255516,0.012303,0.552511,2.162336,1.0,0.006613,1.663694,0.549779,0.046342,0.398928,0.300331
200,"frozenset({yogurt, whipped/sour cream})",frozenset({whole milk}),0.020742,0.255516,0.010880,0.524510,2.052747,1.0,0.005580,1.565719,0.523711,0.040996,0.361316,0.283544
159,"frozenset({rolls/buns, root vegetables})",frozenset({whole milk}),0.024301,0.255516,0.012710,0.523013,2.046888,1.0,0.006500,1.560804,0.524192,0.047583,0.359305,0.286377
64,"frozenset({pip fruit, other vegetables})",frozenset({whole milk}),0.026131,0.255516,0.013523,0.517510,2.025351,1.0,0.006846,1.543003,0.519843,0.050436,0.351913,0.285217


## Question 13

**Question 13 (from lab sheet)**  
*"Find rules whose consequent is `"whole milk"`."*

**Explanation**  
We filter the `rules` DataFrame by checking whether the item `"whole milk"` belongs to each rule's `consequents` set.  
This gives us the rules that **predict** the purchase of `whole milk` given some antecedent.

In [15]:
# Question 13: rules whose consequent contains 'whole milk'

rules_milk = rules[rules['consequents'].apply(lambda x: 'whole milk' in x)]

print("Number of rules with consequent containing 'whole milk':", len(rules_milk))
rules_milk.sort_values(by="confidence", ascending=False).head(20)

Number of rules with consequent containing 'whole milk': 73


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
108,"frozenset({yogurt, curd})",frozenset({whole milk}),0.017285,0.255516,0.010066,0.582353,2.279125,1.0,0.005649,1.782567,0.571107,0.038313,0.439011,0.310874
78,"frozenset({butter, other vegetables})",frozenset({whole milk}),0.020031,0.255516,0.011490,0.573604,2.244885,1.0,0.006371,1.745992,0.565878,0.043512,0.427260,0.309285
54,"frozenset({tropical fruit, root vegetables})",frozenset({whole milk}),0.021047,0.255516,0.011998,0.570048,2.230969,1.0,0.006620,1.731553,0.563627,0.045350,0.422484,0.308502
162,"frozenset({yogurt, root vegetables})",frozenset({whole milk}),0.025826,0.255516,0.014540,0.562992,2.203354,1.0,0.007941,1.703594,0.560625,0.054497,0.413006,0.309948
222,"frozenset({domestic eggs, other vegetables})",frozenset({whole milk}),0.022267,0.255516,0.012303,0.552511,2.162336,1.0,0.006613,1.663694,0.549779,0.046342,0.398928,0.300331
200,"frozenset({yogurt, whipped/sour cream})",frozenset({whole milk}),0.020742,0.255516,0.010880,0.524510,2.052747,1.0,0.005580,1.565719,0.523711,0.040996,0.361316,0.283544
159,"frozenset({rolls/buns, root vegetables})",frozenset({whole milk}),0.024301,0.255516,0.012710,0.523013,2.046888,1.0,0.006500,1.560804,0.524192,0.047583,0.359305,0.286377
64,"frozenset({pip fruit, other vegetables})",frozenset({whole milk}),0.026131,0.255516,0.013523,0.517510,2.025351,1.0,0.006846,1.543003,0.519843,0.050436,0.351913,0.285217
39,"frozenset({yogurt, tropical fruit})",frozenset({whole milk}),0.029283,0.255516,0.015150,0.517361,2.024770,1.0,0.007668,1.542528,0.521384,0.056184,0.351714,0.288326
31,"frozenset({yogurt, other vegetables})",frozenset({whole milk}),0.043416,0.255516,0.022267,0.512881,2.007235,1.0,0.011174,1.528340,0.524577,0.080485,0.345695,0.300014


## Question 14

**Question 14 (from lab sheet)**  
*"Analyze the effect of `min_support` and `min_confidence` on the number of itemsets/rules."*

**Explanation**  
Here we run Apriori (via mlxtend) several times with different `min_support` values and record **how many frequent itemsets** we get.  
Similarly, we could also vary `min_confidence` when generating rules, but the main idea is:

- **Higher `min_support`**  \(\Rightarrow\) **fewer** itemsets (we keep only very common patterns).  
- **Lower `min_support`**  \(\Rightarrow\) **more** itemsets (including rarer patterns, but also more noise and higher runtime).  
- **Higher `min_confidence`**  \(\Rightarrow\) fewer but more reliable rules.  
- **Lower `min_confidence`**  \(\Rightarrow\) more rules, including weak ones.

We will replicate the hint from the lab: try multiple `min_support` values and count how many frequent itemsets are found for each value.

In [16]:
# Question 14: effect of min_support on number of itemsets

supports = [0.01, 0.02, 0.03, 0.05]
counts = []

for s in supports:
    itemsets = mlxtend_apriori(df, min_support=s, use_colnames=True)
    counts.append(len(itemsets))

summary_df = pd.DataFrame({
    "min_support": supports,
    "num_itemsets": counts
})

print("Number of frequent itemsets for different min_support values:")
summary_df

Number of frequent itemsets for different min_support values:


,min_support,num_itemsets
0,0.01,333
1,0.02,122
2,0.03,63
3,0.05,31


### Short discussion of the results

From the table above, we can see the **trade-off**:
- When `min_support` is **low** (e.g. 0.01), we get **many** frequent itemsets, including rare but possibly interesting patterns.
- When `min_support` increases, the number of frequent itemsets drops quickly, because only very common co-occurrences remain.

For association rules (not shown numerically here but conceptually similar):
- Increasing `min_confidence` reduces the number of rules and keeps only strong rules.  
- Decreasing `min_confidence` produces more rules, but some may be weak or not useful in practice.

In real applications, we must choose `min_support` and `min_confidence` carefully to balance **amount of knowledge** vs **noise and runtime**.